### Implementation of casual Attention from first principle approach 
1. Casual attention (Masked attention) is an advancement to self-attention mechanism. 
2. In `casual attention` our focus is only on the current token at or before. All the future tokens should be masked out so that it will not influence model learning. 
3. Each process will be same but we need to masked out the future token which we are not considering after the query value. 

In [1]:
import torch
import torch.nn as nn

### Implementing casual attention with approach 1
- All is same as the self-attention mechanism 
- Masked_attention_matrix = Attention_matrix * Lower triangular matrix
- Calculate context vector with this masked_attention_matrix  

In [15]:
# Implementing casual attention in class module 
class CasualAttentionV1(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias= False):
        super().__init__()
        self.W_q = nn.Linear(d_in, d_out, bias= qkv_bias)
        self.W_k = nn.Linear(d_in, d_out, bias= qkv_bias)
        self.W_v = nn.Linear(d_in, d_out, bias= qkv_bias)
        
    def forward(self, x):
        # Calculating query, key, value matrix 
        query = self.W_q(x)
        keys = self.W_k(x)
        values = self.W_v(x)

        # Attention matrix 
        attention_matrix = torch.softmax(query @ keys.T / keys.shape[-1]**5, dim= -1)

        # having lower triangular matrix with value = 1
        context_length = x.shape[0] # 6 (having 6 tokens in inputs embeddings)
        lower_triangle_matrix = torch.tril(torch.ones(context_length, context_length))

        # Masked attention matrix 
        masked_attention = attention_matrix * lower_triangle_matrix

        context_vector = masked_attention @ values 

        return masked_attention

In [16]:
inputs = torch.tensor([
    [0.43, 0.15, 0.89], # Your
    [0.55, 0.87, 0.66], # Journey
    [0.57, 0.85, 0.64], # Starts
    [0.22, 0.58, 0.33], # With
    [0.77, 0.25, 0.40], # One
    [0.05, 0.80, 0.55], # Step
])

# Creating object of CasualAttentionV1 class 
d_in = inputs.shape[-1]
d_out = 2 # harcoded (This is similar to d_in dimension in GPT, LLM's)
casual_atten_v1 = CasualAttentionV1(d_in, d_out)

masked_att = casual_atten_v1.forward(inputs)
print(f'masked_att.shape: {masked_att.shape}')
print(f'Masked att.values: \n{masked_att}')


masked_att.shape: torch.Size([6, 6])
Masked att.values: 
tensor([[0.1675, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1680, 0.1661, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1680, 0.1661, 0.1661, 0.0000, 0.0000, 0.0000],
        [0.1675, 0.1664, 0.1664, 0.1665, 0.0000, 0.0000],
        [0.1673, 0.1663, 0.1662, 0.1669, 0.1662, 0.0000],
        [0.1679, 0.1663, 0.1663, 0.1663, 0.1671, 0.1662]],
       grad_fn=<MulBackward0>)


#### Implementation of approach 2 
- infinity lower triangle masked matrix + dropout probablity for masked tokens 
    - This improves generalization of model + prevents overfitting + prevents inactivity of neurons in hidden layer 
- Workflow of approach 
![image.png](../image.png)

In [67]:
# Implementation of casual-attention v2 with 2nd approach 
class CasualAttentionV2(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout_value, qkv_bias= False):
        super().__init__()
        self.W_q = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_k = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_v = nn.Linear(d_in, d_out, bias=qkv_bias)

        # Registering buffer, so that creating module will not be impacted by device (cpu, gpu)
        self.register_buffer('masked_values', torch.triu(torch.ones(context_length, context_length), diagonal= 1))

        # Droput layer to select probablity masked values 
        self.dropout = nn.Dropout(dropout_value)
    
    def forward(self, x):
        # Calculating query, key, and value matrices 
        query = self.W_q(x)
        keys = self.W_k(x)
        values = self.W_v(x)

        # Calculating attention scores 
        attention_scores = query @ keys.transpose(1, 2) # row = index(1) = 6, col = index(2) = 2

        # Transforming attention_score matrix into masked_attention_scores
        # Inplace operation to replace out all the upper triangular values to -inf 
        attention_scores = attention_scores.masked_fill_(
            self.masked_values.bool(), -torch.inf
        )

        # Applying softmax and scaling by keys.dims 
        attention_matrix = torch.softmax(
            attention_scores / keys.shape[-1]**0.5, dim= -1 # Row wise sum == 1
        )
        
        # Calculating Dropout values 
        attention_matrix = self.dropout(attention_matrix)

        # Calculating context vector for each token and batches 
        context_vector = attention_matrix @ values

        return context_vector
        

In [68]:
# Calculating context vector batches wise
batch = torch.stack((inputs, inputs)) # Using input embedding as batches 
d_in = batch.shape[-1] # 3 feature dims 
d_out = 2 # Hardcoded value
context_length = batch.shape[-2] # No. of tokens per batch 
dropout = 0.0 # 50% probablistic value
attention_samples = CasualAttentionV2(d_in, d_out, context_length, dropout, qkv_bias= False)


print(attention_samples.forward(batch))

tensor([[[0.3931, 0.0679],
         [0.3848, 0.0791],
         [0.3835, 0.0880],
         [0.3286, 0.0671],
         [0.3492, 0.1154],
         [0.3114, 0.0703]],

        [[0.3931, 0.0679],
         [0.3848, 0.0791],
         [0.3835, 0.0880],
         [0.3286, 0.0671],
         [0.3492, 0.1154],
         [0.3114, 0.0703]]], grad_fn=<UnsafeViewBackward0>)


In [69]:
# --- Step-by-step casual attention on BATCHED inputs ---
# Why batched? In practice, LLMs process multiple sequences at once (a "batch")
# for efficiency. Here we simulate that by stacking the same input twice.

# Shape: (2, 6, 3) → 2 sequences in the batch, each with 6 tokens, each token has 3 features
batch = torch.stack((inputs, inputs))

# --- Step 1: Create learnable Query and Key projection matrices ---
# These linear layers learn to project each 3-dim token into a 2-dim query/key space.
# Think of it as: "What am I looking for?" (query) and "What do I contain?" (key)
weight_query = torch.nn.Linear(batch.shape[-1], 2, bias=False)  # (3 → 2)
weight_key = torch.nn.Linear(batch.shape[-1], 2, bias=False)    # (3 → 2)

# --- Step 2: Project all tokens into query and key representations ---
# Each token in every batch gets its own query and key vector
query_samp = weight_query(batch)  # (2, 6, 2) — 2 batches, 6 tokens, 2-dim query per token
key_samp = weight_key(batch)      # (2, 6, 2) — 2 batches, 6 tokens, 2-dim key per token

# --- Step 3: Compute attention scores (how much each token attends to every other token) ---
# Formula: score = Query @ Key^T
# We transpose the last two dims of key: (2,6,2) → (2,2,6)
# Matrix multiply: (2,6,2) @ (2,2,6) → (2,6,6)
# Result: a 6×6 score matrix per batch — score[i][j] = "how relevant is token j to token i?"
att_score_samp = query_samp @ key_samp.transpose(1, 2)
print(f"Attention score Sample: \n{att_score_samp}")

# --- Step 4: Apply causal mask (the key idea behind "causal" attention) ---
# WHY: During text generation, token at position i should ONLY look at tokens 0..i (past + self).
#       It must NOT peek at future tokens (i+1, i+2, ...) — that would be cheating!
# HOW: We create an upper-triangular matrix of 1s, then replace those positions with -inf.
#       After softmax, -inf becomes 0 → future tokens contribute zero attention.
#
# Example for 6 tokens (1 = will be masked, 0 = allowed):
#   [[0, 1, 1, 1, 1, 1],   ← token 0 can only see itself
#    [0, 0, 1, 1, 1, 1],   ← token 1 can see token 0 and itself
#    [0, 0, 0, 1, 1, 1],   ← token 2 can see tokens 0, 1, and itself
#    [0, 0, 0, 0, 1, 1],   ...
#    [0, 0, 0, 0, 0, 1],
#    [0, 0, 0, 0, 0, 0]]   ← token 5 can see all tokens (nothing masked)

context_length = batch.shape[-2]  # 6 tokens per batch
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)  # upper triangle, diagonal=1 keeps the main diagonal as 0

# masked_fill_: everywhere mask is True (1), replace the score with -inf (in-place)
# This ensures future tokens get zero weight after softmax
att_score_samp = att_score_samp.masked_fill_(mask.bool(), -torch.inf)
print(att_score_samp)

Attention score Sample: 
tensor([[[-0.0189, -0.0439, -0.0429, -0.0274, -0.0171, -0.0383],
         [ 0.2477,  0.3132,  0.3001,  0.1945,  0.0730,  0.3312],
         [ 0.2520,  0.3184,  0.3052,  0.1978,  0.0742,  0.3369],
         [ 0.1465,  0.1888,  0.1811,  0.1173,  0.0453,  0.1982],
         [ 0.1985,  0.2415,  0.2311,  0.1500,  0.0531,  0.2593],
         [ 0.1131,  0.1478,  0.1419,  0.0918,  0.0361,  0.1544]],

        [[-0.0189, -0.0439, -0.0429, -0.0274, -0.0171, -0.0383],
         [ 0.2477,  0.3132,  0.3001,  0.1945,  0.0730,  0.3312],
         [ 0.2520,  0.3184,  0.3052,  0.1978,  0.0742,  0.3369],
         [ 0.1465,  0.1888,  0.1811,  0.1173,  0.0453,  0.1982],
         [ 0.1985,  0.2415,  0.2311,  0.1500,  0.0531,  0.2593],
         [ 0.1131,  0.1478,  0.1419,  0.0918,  0.0361,  0.1544]]],
       grad_fn=<UnsafeViewBackward0>)
tensor([[[-0.0189,    -inf,    -inf,    -inf,    -inf,    -inf],
         [ 0.2477,  0.3132,    -inf,    -inf,    -inf,    -inf],
         [ 0.2520,  0.3